# Outline

2022/06/05

- [ ] Two-pointer templates
- [ ] Typical use cases for two pointers
    - [ ] Sliding window
    - [ ] Fast/slow pointers and cycle problems
    
- Reference link [labuladong](https://labuladong.github.io/algo/2/18/21/)

# Theory

The two-pointer technique is mainly divided into two categories: left-right pointers and fast-slow pointers.

- Left-right pointers: two pointers moving toward each other or away from each other;
- Fast-slow pointers: two pointers moving in the same direction, one fast and one slow.
    - A common fast-slow pointer technique in array problems is to modify the array in place.
    
    - The fast-slow pointer characteristic of the sliding window algorithm: the left pointer trails behind and the right pointer leads; the section between the two pointers is the "window", and the algorithm solves certain problems by expanding and shrinking the "window".


# Patterns: When to Use Two Pointers

## 1. Opposite-direction pointers (`l=0, r=len-1`, move inward)

**Use when:** array/string is **sorted** (or can be sorted) and you're searching for a pair/triplet satisfying a sum/comparison condition.

**Why it works:** at each step you can eliminate one end because moving the other pointer can't possibly do better — that's the key insight that tells you two pointers applies at all, vs needing brute force or a hashmap.

```python
l, r = 0, len(nums) - 1
while l < r:
    s = nums[l] + nums[r]
    if s == target: ...
    elif s < target: l += 1   # need bigger -> move left up
    else: r -= 1               # need smaller -> move right down
```

**Signals:** "sorted array," "pair/triplet that sums to X," "container with most water," "trapping rain water," "3Sum/4Sum" (outer loop + inner two-pointer), palindrome checking (`l==0, r==n-1` comparing inward).

From this notebook's list: 11, 15, 16, 18, 42, 167, 5 (palindrome).

## 2. Fast-slow, same direction (in-place array modification)

**Use when:** you need to **overwrite an array in place** to keep only elements matching some condition, without extra space, preserving order.

**Why it works:** slow = "next write position," fast = "scanner." See cells below (26, 27) — the pattern generalizes to Move Zeroes, Sort Colors (a 3-pointer variant: low/mid/high), Partition List (86), Remove Duplicates II.

```python
slow = 0
for fast in range(len(nums)):
    if keep_condition(nums[fast]):
        nums[slow] = nums[fast]
        slow += 1
return slow
```

## 3. Fast-slow, different speeds (cycle detection)

**Use when:** linked list / functional graph, and you need to detect a cycle or find a midpoint **without extra memory** (the O(n) hashset approach is the "obvious" alternative, so this pattern is specifically the space-optimized answer).

**Signals:** "find duplicate number" (287 — treats array as implicit linked list via indices), "linked list cycle" (141/142 — Floyd's algorithm, then reset one pointer to head to find the cycle start), "find middle of linked list."

```python
slow = fast = head
while fast and fast.next:
    slow = slow.next
    fast = fast.next.next
    if slow == fast: break  # cycle found
```

## 4. Sliding window (fast-slow tracking a *window*, not just positions)

**Use when:** contiguous subarray/substring problem with a constraint that's monotonic — i.e. if the window satisfies the constraint, shrinking it from the left still satisfies it (or the reverse). That monotonicity is the prerequisite; without it, sliding window gives wrong answers and you need prefix sums / DP instead.

**Signals:** "longest/shortest substring/subarray with property X," "at most K distinct," "minimum window containing."

```python
l = 0
for r in range(len(s)):
    add(s[r])                      # expand window
    while window_invalid():        # shrink until valid again
        remove(s[l]); l += 1
    update_answer(r - l + 1)       # window is valid here
```

## Quick decision checklist

| Question | If yes -> |
|---|---|
| Sorted array + looking for pair/triplet by sum? | Opposite-direction pointers |
| Need to compact/filter array in place, no extra space? | Fast-slow, same start |
| Linked list, need O(1) space cycle/midpoint detection? | Fast-slow, different speeds |
| Contiguous subarray/substring, constraint monotonic as window grows/shrinks? | Sliding window |
| None of the above but "two things converge/compare"? | Probably not two pointers -- check hashmap/DP first |


## Worked Example: [(easy) 167 Two Sum II - Input Array Is Sorted](https://leetcode.cn/problems/two-sum-ii-input-array-is-sorted/)

**Problem:** given a 1-indexed array `numbers` sorted in non-decreasing order, find two numbers that add up to `target`. Return their 1-indexed positions `[i, j]` with `i < j`. Exactly one solution exists; you may not use the same element twice.

**Why this is the canonical opposite-direction example:** the array is sorted, and we want a pair with an exact sum -- textbook match for pattern 1 above. It also makes the brute-force -> two-pointer upgrade very concrete: same problem, O(n^2)/O(1) space vs O(n)/O(1) space, just by exploiting sortedness instead of scanning all pairs.

In [ ]:
# 167 -- Brute force: check every pair
# Time: O(n^2) -- nested loop over all (i, j) pairs
# Space: O(1)
from typing import List

class SolutionBruteForce:
    def twoSum(self, numbers: List[int], target: int) -> List[int]:
        n = len(numbers)
        for i in range(n):
            for j in range(i + 1, n):
                if numbers[i] + numbers[j] == target:
                    return [i + 1, j + 1]  # 1-indexed
        return []

print(SolutionBruteForce().twoSum([2, 7, 11, 15], 9))   # [1, 2]
print(SolutionBruteForce().twoSum([2, 3, 4], 6))         # [1, 3]


In [ ]:
# 167 -- Optimal: opposite-direction two pointers
# Time: O(n) -- each pointer moves at most n steps total, never revisits
# Space: O(1)
#
# Key insight: since the array is sorted, if numbers[l] + numbers[r] > target,
# increasing l can only make the sum bigger (worse), so r MUST decrease.
# Symmetrically, if the sum is too small, l MUST increase. No pair is ever
# skipped, because the discarded pointer position can't be part of any
# valid answer once we know which direction the sum needs to move.

class SolutionOptimal:
    def twoSum(self, numbers: List[int], target: int) -> List[int]:
        l, r = 0, len(numbers) - 1
        while l < r:
            s = numbers[l] + numbers[r]
            if s == target:
                return [l + 1, r + 1]  # 1-indexed
            elif s < target:
                l += 1
            else:
                r -= 1
        return []

print(SolutionOptimal().twoSum([2, 7, 11, 15], 9))   # [1, 2]
print(SolutionOptimal().twoSum([2, 3, 4], 6))         # [1, 3]


# LeetCode Example Problems

## Fast-Slow Pointers


- [(easy)26 Remove Duplicates From Sorted Array](https://leetcode.cn/problems/remove-duplicates-from-sorted-array/)
    - Approach: the fast pointer traverses the array, and the slow pointer points to the last element of the duplicate-free array, used to check for equality or to add a new number. The precondition is that the array is sorted. Otherwise you could use a hashmap/hashset. If the range of array values is known, you could also use a 1D array, which sorts as a side effect (counting sort).
    
- [(easy)27 Remove Element](https://leetcode.cn/problems/remove-element/)
    - Approach: fast-slow pointers. The fast pointer traverses the array, the slow pointer points to the next position to fill. If the value is not equal to val, fill it in and move the slow pointer forward by one.
    
    

- [(medium)5 Longest Palindromic Substring](https://leetcode.cn/problems/longest-palindromic-substring/)
- [(medium)3 Longest Substring Without Repeating Characters](https://leetcode.cn/problems/longest-substring-without-repeating-characters)

In [ ]:
# 26: the fast pointer traverses the array, the slow pointer points to the last element of the duplicate-free array, used to check for equality or to add a new number.
# Runtime: 32 ms, beats 98.12% of Python3 submissions
# Memory: 16 MB, beats 71.99% of Python3 submissions
# Important! cannot remove from list in place as I will get out of index error
class Solution:
    def removeDuplicates(self, nums: List[int]) -> int:
        # start with slow / write pointer at index
        last_id = 0
        for num in nums:
            # if current element is not equal to last non duplicate element
            # advance slow pointer by one and move elemnt to that position
            # if element is equal to last non duplicate element i.e. duplicate then skip. 
            # this will be overwritten with next none duplicate element
            if num != nums[last_id]:
                last_id += 1
                nums[last_id] = num
        return last_id + 1


Solution.removeDuplicates(nums=[1,2,2,3])
"""
last_id = 0 num=1 nums=[1,2,2,3] -> no change as 1=1
last_id = 0 num=2 nums=[1,2,2,3] -> 1!= 2, increment last_id and move slow pointer
last_id = 1 num=2 nums=[1,2,2,3]
last_id =2 num=3 nums=[1,2,3,3]
"""

In [ ]:
# 27
# Important! cannot remove from list in place as I will get out of index error
class Solution:
    def removeElement(self, nums: List[int], val: int) -> int:
        # start with slow / write pointer at 0
        last_id = 0
        for num in nums:
            # check if current value equals value to remove
            # if not equal write number to last valid element, advance pointer to next element
            if num != val:
                nums[last_id] = num
                last_id += 1
        return last_id

Solution.removeElement(nums=[1,2,2,3], val=2)
"""
last_id = 0 num=1 nums=[1,2,2,3] -> 1!=2, write 1 to 0, set last_id to 1
last_id = 1 num=2 nums=[1,2,2,3] -> 2 = 2, skip
last_id = 1 num=2 nums=[1,2,2,3] -> 2 =2 skip
last_id =1 num=3 nums=[1,2,2,3] -> 3!=2, write 3 to 1, set last_id to 2
last_id =2 num=3 nums=[1,3,2,3] 
"""

nums = [1,2,5,4,3]
target = 9

def two_sum(nums: list[int], target: int):
    for i in range(len(nums)):
        complement = target - nums[i]
        

    return []


In [ ]:
# 5: enumerate starting from a middle coordinate; enumerate separately for odd and even length
class Solution:
    def longestPalindrome(self, s: str) -> str:
        max_len = 1
        max_l = 0
        max_r = 0
        l = len(s)
        for mid in range(l):
            for i in range(1, l):
                if mid - i < 0 or mid + i >= l or s[mid-i] != s[mid + i]:
                    if 2 * i - 1 > max_len:
                        max_l = mid - i + 1
                        max_r = mid + i - 1
                        max_len = 2 * i - 1
                    break
            if mid < l - 1 and s[mid] == s[mid + 1]:
                for i in range(l):
                    if mid - i < 0 or mid + 1 + i >= l or s[mid-i] != s[mid + 1 + i]:
                        if 2 * i > max_len:
                            max_l = mid - i + 1
                            max_r = mid + i 
                            max_len = 2 * i
                        break
        return s[max_l : (max_r + 1)]





## Sliding Window

- [(medium)3 Longest Substring Without Repeating Characters](https://leetcode.cn/problems/longest-substring-without-repeating-characters)
- [(medium)396 Rotate Function](https://leetcode-cn.com/problems/rotate-function/)
- [(easy)643 Maximum Average Subarray I](https://leetcode-cn.com/problems/maximum-average-subarray-i/)
- [(easy)1984 Minimum Difference Highest and Lowest of K Scores](https://leetcode-cn.com/problems/minimum-difference-between-highest-and-lowest-of-k-scores/)
- [(medium)567 Permutation in String](https://leetcode.cn/problems/permutation-in-string)
- [(hard)480 Sliding Window Median](https://leetcode.cn/problems/sliding-window-median)
- [(hard)76 Minimum Window Substring](https://leetcode.cn/problems/minimum-window-substring)
- [(hard)239 Sliding Window Maximum](https://leetcode.cn/problems/sliding-window-maximum)
- [(medium)978 Longest Turbulent Subarray](https://leetcode.cn/problems/longest-turbulent-subarray)
- [(hard)992 Subarrys with K Different Integers](https://leetcode.cn/problems/subarrays-with-k-different-integers)
- [(hard)995 Minimum Number of K Consecutive Bit Flips](https://leetcode.cn/problems/minimum-number-of-k-consecutive-bit-flips)

## Uncategorized


- [(medium)15 3 Sum](https://leetcode.cn/problems/3sum)
- [(medium)16 3 Sum CLosest](https://leetcode.cn/problems/3sum-closest)
- [(medium)18 4 Sum](https://leetcode.cn/problems/4sum)
- [(hard)42 Trapping Rain Water](https://leetcode.cn/problems/trapping-rain-water)
- [(medium)75 Sort Colors](https://leetcode.cn/problems/sort-colors)
- [(medium)86 Partition List](https://leetcode.cn/problems/partition-list)
- [(easy)88 Merge Sorted Array](https://leetcode.cn/problems/merge-sorted-array)
- [(easy)141 Linked List Cycle](https://leetcode.cn/problems/linked-list-cycle)
- [(medium)142 Linked List Cycle II](https://leetcode.cn/problems/linked-list-cycle-ii)
- [(medium)287 Find the Duplicate Number](https://leetcode.cn/problems/find-the-duplicate-number)
- [(medium)763 Partition Labels](https://leetcode.cn/problems/partition-labels)
- [(medium)11 Container With Most Water](https://leetcode.cn/problems/container-with-most-water)
- [(medium)19 Remove Nth Node From End of List](https://leetcode.cn/problems/remove-nth-node-from-end-of-list)
- [(easy)1446 Consecutive Characters](https://leetcode.cn/problems/consecutive-characters)




# Lessons Learned

The classic pattern for the two-pointer sliding window: the right pointer keeps moving right until it can no longer do so (the exact condition depends on the problem). Once the right pointer reaches the far right, start moving the left pointer to release the left boundary of the window. Problems 3, 76, 209, 424, 438, 567, 713, 763, 845, 881, 904, 978, 992, 1004, 1040, 1052.

    Fast-slow pointers can find a duplicate number in O(n) time. Problem 287.
    After replacing letters, the longest run of the same consecutive letter. Problem 424.
    SUM problem set. Problems 1, 15, 16, 18, 167, 923, 1074.